# GPU Utilization Debug Notebook for GraphSAGE Training

This notebook helps identify and resolve GPU underutilization when training GraphSAGE models.

**Sections:**
1. Environment and Baseline Profiling
2. Data Loading Analysis
3. Batch Analysis
4. Model Forward Pass Profiling
5. Optimized Training Loop
6. Summary and Recommendations

## Section 1: Environment and Baseline Profiling

In [ ]:
# =============================================================================
# IMPORTS AND PATH SETUP
# =============================================================================

import sys
import time
import gc
from pathlib import Path
from datetime import datetime

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('.')

sys.path.insert(0, str(BASE_PATH))

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool

# Import from models.py
from models import (
    GraphSAGEModel,
    GraphSAGE,
    DatasetCache,
)

print(f"BASE_PATH: {BASE_PATH}")
print(f"Running in Colab: {IN_COLAB}")

In [ ]:
# =============================================================================
# GPU DIAGNOSTICS
# =============================================================================

def print_gpu_info():
    """Print comprehensive GPU information."""
    print("="*60)
    print("GPU DIAGNOSTICS")
    print("="*60)
    
    if not torch.cuda.is_available():
        print("CUDA is NOT available! Training will be on CPU.")
        return False
    
    print(f"\nCUDA Available: {torch.cuda.is_available()}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
    print(f"PyTorch Version: {torch.__version__}")
    
    print(f"\n--- GPU Device Info ---")
    print(f"Device Count: {torch.cuda.device_count()}")
    print(f"Current Device: {torch.cuda.current_device()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    
    # Get device properties
    props = torch.cuda.get_device_properties(0)
    print(f"\n--- GPU Specifications ---")
    print(f"Total Memory: {props.total_memory / 1024**3:.2f} GB")
    print(f"Multi-Processor Count: {props.multi_processor_count}")
    print(f"Compute Capability: {props.major}.{props.minor}")
    
    # Memory info
    print(f"\n--- Current Memory Usage ---")
    print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.3f} GB")
    print(f"Cached: {torch.cuda.memory_reserved(0) / 1024**3:.3f} GB")
    
    # Check if TF32 is enabled (Ampere+ GPUs)
    print(f"\n--- Performance Settings ---")
    print(f"TF32 for MatMul: {torch.backends.cuda.matmul.allow_tf32}")
    print(f"TF32 for cuDNN: {torch.backends.cudnn.allow_tf32}")
    print(f"cuDNN Benchmark: {torch.backends.cudnn.benchmark}")
    
    return True

gpu_available = print_gpu_info()
device = torch.device("cuda" if gpu_available else "cpu")
print(f"\nUsing device: {device}")

In [ ]:
# =============================================================================
# ENABLE PERFORMANCE OPTIMIZATIONS
# =============================================================================

if torch.cuda.is_available():
    # Enable TF32 for faster computation on Ampere+ GPUs (RTX 30xx, A100, etc.)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    # Enable cuDNN autotuner - finds optimal algorithms for your specific GPU
    torch.backends.cudnn.benchmark = True
    
    print("Performance optimizations enabled:")
    print(f"  TF32 MatMul: {torch.backends.cuda.matmul.allow_tf32}")
    print(f"  TF32 cuDNN: {torch.backends.cudnn.allow_tf32}")
    print(f"  cuDNN Benchmark: {torch.backends.cudnn.benchmark}")

In [ ]:
# =============================================================================
# GPU UTILIZATION MONITORING UTILITIES
# =============================================================================

class GPUMonitor:
    """Simple GPU utilization tracker."""
    
    def __init__(self):
        self.has_pynvml = False
        try:
            import pynvml
            pynvml.nvmlInit()
            self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)
            self.has_pynvml = True
            print("pynvml initialized - GPU utilization monitoring available")
        except:
            print("pynvml not available - install with: pip install pynvml")
            print("GPU utilization will be estimated from memory usage only")
    
    def get_utilization(self):
        """Get GPU utilization percentage."""
        if self.has_pynvml:
            import pynvml
            util = pynvml.nvmlDeviceGetUtilizationRates(self.handle)
            return {'gpu': util.gpu, 'memory': util.memory}
        return None
    
    def get_memory_info(self):
        """Get GPU memory info."""
        if torch.cuda.is_available():
            return {
                'allocated_gb': torch.cuda.memory_allocated(0) / 1024**3,
                'cached_gb': torch.cuda.memory_reserved(0) / 1024**3,
                'max_allocated_gb': torch.cuda.max_memory_allocated(0) / 1024**3,
            }
        return None
    
    def reset_peak_memory(self):
        """Reset peak memory tracking."""
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats(0)

gpu_monitor = GPUMonitor()

In [ ]:
# =============================================================================
# TIMING UTILITIES
# =============================================================================

class CUDATimer:
    """Accurate GPU timing using CUDA events."""
    
    def __init__(self):
        self.use_cuda = torch.cuda.is_available()
        if self.use_cuda:
            self.start_event = torch.cuda.Event(enable_timing=True)
            self.end_event = torch.cuda.Event(enable_timing=True)
        self.cpu_start = None
    
    def start(self):
        if self.use_cuda:
            self.start_event.record()
        self.cpu_start = time.perf_counter()
    
    def stop(self):
        cpu_time = time.perf_counter() - self.cpu_start
        if self.use_cuda:
            self.end_event.record()
            torch.cuda.synchronize()
            gpu_time = self.start_event.elapsed_time(self.end_event) / 1000.0  # ms to s
            return {'gpu_time': gpu_time, 'cpu_time': cpu_time}
        return {'cpu_time': cpu_time}

def benchmark_fn(fn, n_runs=10, warmup=3):
    """Benchmark a function with proper warmup."""
    timer = CUDATimer()
    
    # Warmup
    for _ in range(warmup):
        fn()
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    # Benchmark
    times = []
    for _ in range(n_runs):
        timer.start()
        fn()
        result = timer.stop()
        times.append(result.get('gpu_time', result['cpu_time']))
    
    return {
        'mean': np.mean(times),
        'std': np.std(times),
        'min': np.min(times),
        'max': np.max(times),
    }

print("Timing utilities ready.")

## Section 2: Data Loading Analysis

This section profiles:
- Dataset loading time from cache
- DataLoader iteration speed with different `num_workers`
- Impact of `persistent_workers` and `prefetch_factor`

In [ ]:
# =============================================================================
# LOAD DATASET FROM CACHE
# =============================================================================

# Choose which dataset to use for debugging (d3 is fastest, d7 is more representative)
DATASET_NAME = "d5_baseline"  # Options: d3_baseline, d5_baseline, d7_baseline, d9_baseline

print(f"Loading dataset: {DATASET_NAME}")
print("="*60)

# Time the dataset loading
load_start = time.perf_counter()

cache = DatasetCache(base_path=BASE_PATH, device=device)
cache.load(DATASET_NAME)
graphs = cache.get_graphs()

load_time = time.perf_counter() - load_start

print(f"\nDataset loaded in {load_time:.2f} seconds")
print(f"Total graphs: {len(graphs):,}")
print(f"Loading speed: {len(graphs) / load_time:,.0f} graphs/second")

In [ ]:
# =============================================================================
# BENCHMARK DATALOADER WITH DIFFERENT num_workers
# =============================================================================

def benchmark_dataloader(graphs, batch_size, num_workers, n_batches=100, 
                         persistent_workers=False, prefetch_factor=2):
    """
    Benchmark DataLoader iteration speed.
    
    Returns time to iterate through n_batches.
    """
    loader_kwargs = {
        'batch_size': batch_size,
        'shuffle': True,
        'num_workers': num_workers,
        'pin_memory': True if torch.cuda.is_available() else False,
    }
    
    # persistent_workers only valid when num_workers > 0
    if num_workers > 0:
        loader_kwargs['persistent_workers'] = persistent_workers
        loader_kwargs['prefetch_factor'] = prefetch_factor
    
    loader = DataLoader(graphs, **loader_kwargs)
    
    # Warmup: iterate through a few batches
    loader_iter = iter(loader)
    for _ in range(min(5, n_batches)):
        try:
            batch = next(loader_iter)
            if torch.cuda.is_available():
                batch = batch.to(device)
        except StopIteration:
            loader_iter = iter(loader)
    
    # Benchmark
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start = time.perf_counter()
    batch_count = 0
    total_graphs = 0
    
    loader_iter = iter(loader)
    while batch_count < n_batches:
        try:
            batch = next(loader_iter)
            if torch.cuda.is_available():
                batch = batch.to(device, non_blocking=True)
            batch_count += 1
            total_graphs += batch.num_graphs
        except StopIteration:
            loader_iter = iter(loader)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    elapsed = time.perf_counter() - start
    
    return {
        'time': elapsed,
        'batches_per_sec': batch_count / elapsed,
        'graphs_per_sec': total_graphs / elapsed,
    }

# Test different num_workers configurations
print("="*60)
print("DATALOADER BENCHMARK")
print("="*60)
print(f"\nBatch size: 256, Testing {100} batches each\n")

# Use a subset for faster benchmarking
subset_size = min(100000, len(graphs))
benchmark_graphs = graphs[:subset_size]
print(f"Using {subset_size:,} graphs for benchmarking\n")

worker_configs = [0, 2, 4, 8]
results = []

for num_workers in worker_configs:
    print(f"Testing num_workers={num_workers}...", end=" ", flush=True)
    try:
        result = benchmark_dataloader(
            benchmark_graphs, 
            batch_size=256, 
            num_workers=num_workers,
            n_batches=100,
            persistent_workers=(num_workers > 0),
            prefetch_factor=2
        )
        result['num_workers'] = num_workers
        results.append(result)
        print(f"{result['batches_per_sec']:.1f} batches/sec, {result['graphs_per_sec']:,.0f} graphs/sec")
    except Exception as e:
        print(f"Failed: {e}")

# Find best configuration
if results:
    best = max(results, key=lambda x: x['graphs_per_sec'])
    baseline = results[0]  # num_workers=0
    print(f"\n{'='*60}")
    print(f"RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"Best: num_workers={best['num_workers']} -> {best['graphs_per_sec']:,.0f} graphs/sec")
    print(f"Speedup vs num_workers=0: {best['graphs_per_sec'] / baseline['graphs_per_sec']:.2f}x")

In [ ]:
# =============================================================================
# VISUALIZE DATALOADER RESULTS
# =============================================================================

if results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    workers = [r['num_workers'] for r in results]
    graphs_per_sec = [r['graphs_per_sec'] for r in results]
    batches_per_sec = [r['batches_per_sec'] for r in results]
    
    # Bar chart for graphs/sec
    bars = ax1.bar(range(len(workers)), graphs_per_sec, color='steelblue')
    ax1.set_xticks(range(len(workers)))
    ax1.set_xticklabels([str(w) for w in workers])
    ax1.set_xlabel('num_workers')
    ax1.set_ylabel('Graphs per second')
    ax1.set_title('DataLoader Throughput vs num_workers')
    
    # Add value labels
    for bar, val in zip(bars, graphs_per_sec):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, 
                 f'{val:,.0f}', ha='center', va='bottom', fontsize=9)
    
    # Speedup chart
    baseline = graphs_per_sec[0]
    speedups = [g / baseline for g in graphs_per_sec]
    bars2 = ax2.bar(range(len(workers)), speedups, color='seagreen')
    ax2.set_xticks(range(len(workers)))
    ax2.set_xticklabels([str(w) for w in workers])
    ax2.set_xlabel('num_workers')
    ax2.set_ylabel('Speedup vs num_workers=0')
    ax2.set_title('DataLoader Speedup')
    ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.5)
    
    for bar, val in zip(bars2, speedups):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
                 f'{val:.2f}x', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print("\n⚠️  FINDING: If num_workers=0 is nearly as fast as num_workers>0,")
    print("   the bottleneck is NOT data loading but likely the model forward pass.")

## Section 3: Batch Analysis

This section analyzes:
- Graph sizes (nodes and edges per graph)
- Impact of batch size on GPU utilization
- Memory usage vs batch size tradeoffs

In [ ]:
# =============================================================================
# ANALYZE GRAPH STATISTICS
# =============================================================================

print("="*60)
print("GRAPH STATISTICS ANALYSIS")
print("="*60)

# Sample graphs for analysis
n_sample = min(10000, len(graphs))
sample_graphs = graphs[:n_sample]

# Collect statistics
node_counts = []
edge_counts = []
empty_graphs = 0

for g in tqdm(sample_graphs, desc="Analyzing graphs"):
    n_nodes = g.x.shape[0]
    n_edges = g.edge_index.shape[1] if g.edge_index is not None else 0
    node_counts.append(n_nodes)
    edge_counts.append(n_edges)
    if n_nodes == 0:
        empty_graphs += 1

node_counts = np.array(node_counts)
edge_counts = np.array(edge_counts)

print(f"\nAnalyzed {n_sample:,} graphs")
print(f"\n--- Node Statistics ---")
print(f"  Min nodes: {node_counts.min()}")
print(f"  Max nodes: {node_counts.max()}")
print(f"  Mean nodes: {node_counts.mean():.1f}")
print(f"  Median nodes: {np.median(node_counts):.1f}")
print(f"  Std nodes: {node_counts.std():.1f}")
print(f"  Empty graphs (0 nodes): {empty_graphs} ({100*empty_graphs/n_sample:.1f}%)")

print(f"\n--- Edge Statistics ---")
print(f"  Min edges: {edge_counts.min()}")
print(f"  Max edges: {edge_counts.max()}")
print(f"  Mean edges: {edge_counts.mean():.1f}")
print(f"  Median edges: {np.median(edge_counts):.1f}")

# Estimate memory per graph
avg_nodes = node_counts.mean()
avg_edges = edge_counts.mean()
# Features: float32 [nodes, 5] = nodes * 5 * 4 bytes
# Edge index: int64 [2, edges] = 2 * edges * 8 bytes
# Edge attr: float32 [edges, 1] = edges * 4 bytes
bytes_per_graph = avg_nodes * 5 * 4 + 2 * avg_edges * 8 + avg_edges * 4
print(f"\n--- Memory Estimate ---")
print(f"  Avg memory per graph: {bytes_per_graph / 1024:.1f} KB")
print(f"  Batch of 256: {256 * bytes_per_graph / 1024**2:.1f} MB")
print(f"  Batch of 1024: {1024 * bytes_per_graph / 1024**2:.1f} MB")

In [ ]:
# =============================================================================
# VISUALIZE GRAPH SIZE DISTRIBUTION
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Node count distribution
ax1 = axes[0]
ax1.hist(node_counts, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(node_counts.mean(), color='red', linestyle='--', label=f'Mean: {node_counts.mean():.1f}')
ax1.set_xlabel('Number of Nodes')
ax1.set_ylabel('Frequency')
ax1.set_title('Node Count Distribution')
ax1.legend()

# Edge count distribution
ax2 = axes[1]
ax2.hist(edge_counts, bins=50, color='seagreen', edgecolor='black', alpha=0.7)
ax2.axvline(edge_counts.mean(), color='red', linestyle='--', label=f'Mean: {edge_counts.mean():.1f}')
ax2.set_xlabel('Number of Edges')
ax2.set_ylabel('Frequency')
ax2.set_title('Edge Count Distribution')
ax2.legend()

# Nodes vs Edges scatter
ax3 = axes[2]
ax3.scatter(node_counts[:1000], edge_counts[:1000], alpha=0.3, s=10)
ax3.set_xlabel('Number of Nodes')
ax3.set_ylabel('Number of Edges')
ax3.set_title('Nodes vs Edges (sample)')

plt.tight_layout()
plt.show()

# Insights
print("\n⚠️  INSIGHT:")
if node_counts.mean() < 20:
    print("   Graphs have VERY FEW nodes on average. This means:")
    print("   - Each graph does minimal computation")
    print("   - LARGER batch sizes are essential to keep GPU busy")
    print("   - Consider batch_size >= 1024 or even 2048")
else:
    print("   Graphs have moderate size. Standard batch sizes (256-512) should work.")

In [ ]:
# =============================================================================
# BENCHMARK DIFFERENT BATCH SIZES
# =============================================================================

print("="*60)
print("BATCH SIZE BENCHMARK")
print("="*60)

# Initialize a model for benchmarking
model = GraphSAGEModel(
    in_channels=5,
    hidden_dim=128,
    num_layers=4,
    dropout=0.0,
    aggr='mean'
).to(device)
model.eval()

def benchmark_batch_size(graphs, model, batch_size, n_batches=50, num_workers=4):
    """Benchmark training throughput at a given batch size."""
    loader = DataLoader(
        graphs,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers if num_workers > 0 else 0,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
    )
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = torch.nn.BCELoss()
    
    # Warmup
    model.train()
    loader_iter = iter(loader)
    for _ in range(3):
        try:
            batch = next(loader_iter)
            batch = batch.to(device)
            pred = model(batch)
            loss = loss_fn(pred, batch.y.float().view(-1, 1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        except StopIteration:
            loader_iter = iter(loader)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        gpu_monitor.reset_peak_memory()
    
    # Benchmark
    start = time.perf_counter()
    batch_count = 0
    total_graphs = 0
    
    loader_iter = iter(loader)
    while batch_count < n_batches:
        try:
            batch = next(loader_iter)
            batch = batch.to(device, non_blocking=True)
            pred = model(batch)
            loss = loss_fn(pred, batch.y.float().view(-1, 1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            batch_count += 1
            total_graphs += batch.num_graphs
        except StopIteration:
            loader_iter = iter(loader)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    elapsed = time.perf_counter() - start
    mem_info = gpu_monitor.get_memory_info()
    
    return {
        'time': elapsed,
        'graphs_per_sec': total_graphs / elapsed,
        'batches_per_sec': batch_count / elapsed,
        'peak_memory_gb': mem_info['max_allocated_gb'] if mem_info else 0,
    }

# Test different batch sizes
batch_sizes = [64, 128, 256, 512, 1024, 2048]
batch_results = []

# Determine best num_workers from previous test
best_workers = 4  # Default, update based on Section 2 results

print(f"\nUsing num_workers={best_workers}")
print(f"Testing {50} batches per configuration\n")

for bs in batch_sizes:
    print(f"Batch size {bs}...", end=" ", flush=True)
    try:
        result = benchmark_batch_size(benchmark_graphs, model, bs, n_batches=50, num_workers=best_workers)
        result['batch_size'] = bs
        batch_results.append(result)
        print(f"{result['graphs_per_sec']:,.0f} graphs/sec, Peak mem: {result['peak_memory_gb']:.2f} GB")
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"OOM! GPU memory exceeded.")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        else:
            print(f"Error: {e}")
        break

# Summary
if batch_results:
    best_bs = max(batch_results, key=lambda x: x['graphs_per_sec'])
    print(f"\n{'='*60}")
    print(f"BATCH SIZE RESULTS")
    print(f"{'='*60}")
    print(f"Best: batch_size={best_bs['batch_size']} -> {best_bs['graphs_per_sec']:,.0f} graphs/sec")
    print(f"Peak memory at best: {best_bs['peak_memory_gb']:.2f} GB")

In [ ]:
# =============================================================================
# VISUALIZE BATCH SIZE RESULTS
# =============================================================================

if batch_results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    bs_values = [r['batch_size'] for r in batch_results]
    throughput = [r['graphs_per_sec'] for r in batch_results]
    memory = [r['peak_memory_gb'] for r in batch_results]
    
    # Throughput vs batch size
    ax1.plot(bs_values, throughput, 'o-', color='steelblue', linewidth=2, markersize=8)
    ax1.set_xlabel('Batch Size')
    ax1.set_ylabel('Graphs per second')
    ax1.set_title('Training Throughput vs Batch Size')
    ax1.set_xscale('log', base=2)
    ax1.grid(True, alpha=0.3)
    
    # Memory vs batch size
    ax2.plot(bs_values, memory, 's-', color='coral', linewidth=2, markersize=8)
    ax2.set_xlabel('Batch Size')
    ax2.set_ylabel('Peak GPU Memory (GB)')
    ax2.set_title('GPU Memory Usage vs Batch Size')
    ax2.set_xscale('log', base=2)
    ax2.grid(True, alpha=0.3)
    
    # Add GPU memory limit line if available
    if torch.cuda.is_available():
        total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
        ax2.axhline(y=total_mem * 0.9, color='red', linestyle='--', 
                    label=f'90% GPU mem ({total_mem*0.9:.1f} GB)')
        ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Recommendation
    print("\n⚠️  RECOMMENDATION:")
    print(f"   Use batch_size={best_bs['batch_size']} for optimal throughput")
    print(f"   This achieves {best_bs['graphs_per_sec']:,.0f} graphs/sec")

## Section 4: Model Forward Pass Profiling

This section profiles:
- Bottleneck identification in the forward pass
- Manual pooling loop vs vectorized `global_mean_pool`
- torch.compile() optimization impact

In [ ]:
# =============================================================================
# POOLING COMPARISON: MANUAL LOOP VS VECTORIZED
# =============================================================================

print("="*60)
print("POOLING BENCHMARK: Manual Loop vs global_mean_pool")
print("="*60)

def manual_global_mean_pool(x, batch):
    """
    Current implementation from models.py - uses Python for-loop.
    This is the SLOW version.
    """
    batch_size = int(batch.max().item()) + 1
    x_pooled = torch.zeros(batch_size, x.size(1), device=x.device)
    for i in range(batch_size):
        mask = (batch == i)
        if mask.sum() > 0:
            x_pooled[i] = x[mask].mean(dim=0)
    return x_pooled

def vectorized_global_mean_pool(x, batch):
    """
    PyTorch Geometric's optimized implementation.
    Uses scatter operations - much faster!
    """
    return global_mean_pool(x, batch)

# Create test data
def create_test_batch(batch_size, avg_nodes=20, hidden_dim=128):
    """Create a synthetic batched graph for testing."""
    total_nodes = batch_size * avg_nodes
    x = torch.randn(total_nodes, hidden_dim, device=device)
    batch = torch.repeat_interleave(
        torch.arange(batch_size, device=device),
        avg_nodes
    )
    return x, batch

# Benchmark at different batch sizes
test_batch_sizes = [64, 256, 512, 1024]
pooling_results = []

print(f"\nBenchmarking with hidden_dim=128, avg_nodes=20")
print(f"Each benchmark: 10 runs after 3 warmup runs\n")

for bs in test_batch_sizes:
    x, batch = create_test_batch(bs, avg_nodes=20, hidden_dim=128)
    
    # Benchmark manual pooling
    manual_time = benchmark_fn(lambda: manual_global_mean_pool(x, batch), n_runs=10, warmup=3)
    
    # Benchmark vectorized pooling
    vectorized_time = benchmark_fn(lambda: vectorized_global_mean_pool(x, batch), n_runs=10, warmup=3)
    
    speedup = manual_time['mean'] / vectorized_time['mean']
    
    pooling_results.append({
        'batch_size': bs,
        'manual_ms': manual_time['mean'] * 1000,
        'vectorized_ms': vectorized_time['mean'] * 1000,
        'speedup': speedup
    })
    
    print(f"Batch {bs:4d}: Manual={manual_time['mean']*1000:.3f}ms, "
          f"Vectorized={vectorized_time['mean']*1000:.3f}ms, "
          f"Speedup={speedup:.1f}x")

# Summary
avg_speedup = np.mean([r['speedup'] for r in pooling_results])
print(f"\n{'='*60}")
print(f"POOLING BENCHMARK RESULTS")
print(f"{'='*60}")
print(f"Average speedup from vectorized pooling: {avg_speedup:.1f}x")
print(f"\n⚠️  The manual for-loop in GraphSAGEModel is a MAJOR bottleneck!")
print(f"   Replace it with global_mean_pool() for significant speedup.")

In [ ]:
# =============================================================================
# OPTIMIZED GRAPHSAGE MODEL WITH VECTORIZED POOLING
# =============================================================================

from models import WeightedSAGEConv

class OptimizedGraphSAGEModel(torch.nn.Module):
    """
    GraphSAGE model with VECTORIZED global_mean_pool instead of manual loop.
    
    This is the key optimization for GPU utilization!
    """
    
    def __init__(self, in_channels: int = 5, hidden_dim: int = 128, num_layers: int = 4,
                 dropout: float = 0.0, aggr: str = 'mean'):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout_rate = dropout
        self.aggr = aggr

        # WeightedSAGEConv layers
        self.convs = torch.nn.ModuleList()
        self.convs.append(WeightedSAGEConv(in_channels, hidden_dim, aggr_type=aggr))
        for _ in range(num_layers - 1):
            self.convs.append(WeightedSAGEConv(hidden_dim, hidden_dim, aggr_type=aggr))

        # Batch normalization layers
        self.bns = torch.nn.ModuleList()
        for _ in range(num_layers):
            self.bns.append(torch.nn.BatchNorm1d(hidden_dim))

        # Dropout layer
        self.dropout = torch.nn.Dropout(dropout)

        # Classification head
        self.fc1 = torch.nn.Linear(hidden_dim, 64)
        self.fc2 = torch.nn.Linear(64, 1)

    def forward(self, data) -> torch.Tensor:
        x, edge_index = data.x, data.edge_index
        edge_weight = data.edge_attr.view(-1) if hasattr(data, 'edge_attr') and data.edge_attr is not None else None
        batch = data.batch if hasattr(data, 'batch') and data.batch is not None else torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        # Apply WeightedSAGEConv layers
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_weight)
            x = bn(x)
            x = F.silu(x)
            x = self.dropout(x)

        # OPTIMIZED: Use vectorized global_mean_pool instead of manual loop!
        x_pooled = global_mean_pool(x, batch)

        # Classification layers
        x = self.fc1(x_pooled)
        x = F.silu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = torch.sigmoid(x)

        return x

print("OptimizedGraphSAGEModel defined with vectorized global_mean_pool")

In [ ]:
# =============================================================================
# COMPARE ORIGINAL VS OPTIMIZED MODEL
# =============================================================================

print("="*60)
print("MODEL COMPARISON: Original vs Optimized")
print("="*60)

# Create both models
original_model = GraphSAGEModel(
    in_channels=5, hidden_dim=128, num_layers=4, dropout=0.0, aggr='mean'
).to(device)

optimized_model = OptimizedGraphSAGEModel(
    in_channels=5, hidden_dim=128, num_layers=4, dropout=0.0, aggr='mean'
).to(device)

# Copy weights from original to optimized (ensure same starting point)
optimized_model.load_state_dict(original_model.state_dict())

# Create test batches
test_batch_size = 512
loader = DataLoader(benchmark_graphs[:10000], batch_size=test_batch_size, shuffle=False)
test_batch = next(iter(loader)).to(device)

print(f"\nTest batch: {test_batch.num_graphs} graphs, {test_batch.x.shape[0]} total nodes")

# Benchmark original model forward pass
original_model.eval()
with torch.no_grad():
    original_time = benchmark_fn(lambda: original_model(test_batch), n_runs=20, warmup=5)

# Benchmark optimized model forward pass  
optimized_model.eval()
with torch.no_grad():
    optimized_time = benchmark_fn(lambda: optimized_model(test_batch), n_runs=20, warmup=5)

print(f"\n--- Forward Pass Timing ---")
print(f"Original model:  {original_time['mean']*1000:.3f}ms ± {original_time['std']*1000:.3f}ms")
print(f"Optimized model: {optimized_time['mean']*1000:.3f}ms ± {optimized_time['std']*1000:.3f}ms")
print(f"Speedup: {original_time['mean'] / optimized_time['mean']:.2f}x")

# Verify outputs match
with torch.no_grad():
    orig_out = original_model(test_batch)
    opt_out = optimized_model(test_batch)
    max_diff = (orig_out - opt_out).abs().max().item()
    print(f"\nOutput difference (should be ~0): {max_diff:.6f}")

In [ ]:
# =============================================================================
# TORCH.COMPILE() OPTIMIZATION
# =============================================================================

print("="*60)
print("TORCH.COMPILE() BENCHMARK")
print("="*60)

# Check PyTorch version
torch_version = tuple(map(int, torch.__version__.split('.')[:2]))
if torch_version < (2, 0):
    print(f"torch.compile() requires PyTorch 2.0+. You have {torch.__version__}")
    print("Skipping torch.compile() benchmark.")
    compiled_available = False
else:
    compiled_available = True
    print(f"PyTorch {torch.__version__} supports torch.compile()")

if compiled_available:
    # Create a fresh optimized model
    compiled_model = OptimizedGraphSAGEModel(
        in_channels=5, hidden_dim=128, num_layers=4, dropout=0.0, aggr='mean'
    ).to(device)
    compiled_model.load_state_dict(optimized_model.state_dict())
    
    try:
        # Compile the model
        print("\nCompiling model (first run is slow)...")
        compiled_model = torch.compile(compiled_model, mode='reduce-overhead')
        
        # Warmup (compilation happens here)
        compiled_model.eval()
        with torch.no_grad():
            for _ in range(10):
                _ = compiled_model(test_batch)
        
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        
        # Benchmark compiled model
        with torch.no_grad():
            compiled_time = benchmark_fn(lambda: compiled_model(test_batch), n_runs=20, warmup=5)
        
        print(f"\n--- Forward Pass Timing (with compile) ---")
        print(f"Optimized model:          {optimized_time['mean']*1000:.3f}ms")
        print(f"Optimized + compiled:     {compiled_time['mean']*1000:.3f}ms")
        print(f"Compile speedup: {optimized_time['mean'] / compiled_time['mean']:.2f}x")
        print(f"\nTotal speedup vs original: {original_time['mean'] / compiled_time['mean']:.2f}x")
        
    except Exception as e:
        print(f"torch.compile() failed: {e}")
        print("This can happen with custom layers. Vectorized pooling alone provides big gains.")

## Section 5: Optimized Training Loop

This section implements:
- Complete optimized training function with all improvements
- Mixed precision training (AMP) for additional speedup
- Side-by-side comparison of baseline vs optimized throughput

In [ ]:
# =============================================================================
# OPTIMIZED TRAINING FUNCTION
# =============================================================================

def train_optimized(
    model,
    graphs,
    epochs=1,
    batch_size=512,
    lr=1e-3,
    num_workers=4,
    use_amp=True,
    use_compile=False,
    verbose=True
):
    """
    Optimized training loop with all GPU utilization improvements.
    
    Optimizations:
    1. Multi-worker data loading (num_workers > 0)
    2. Persistent workers for reduced overhead
    3. Pin memory + non_blocking transfers
    4. Mixed precision training (AMP)
    5. Optional torch.compile()
    
    Returns training statistics.
    """
    # Optionally compile the model
    if use_compile and hasattr(torch, 'compile'):
        if verbose:
            print("Compiling model...")
        model = torch.compile(model, mode='reduce-overhead')
    
    # DataLoader with optimizations
    loader = DataLoader(
        graphs,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers if num_workers > 0 else 0,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2 if num_workers > 0 else None,
    )
    
    # Optimizer and loss
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.BCELoss()
    
    # Mixed precision scaler
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp and torch.cuda.is_available())
    
    # Training metrics
    total_graphs = 0
    total_batches = 0
    epoch_losses = []
    
    model.train()
    
    # Reset memory stats
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    
    start_time = time.perf_counter()
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        epoch_graphs = 0
        
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}", disable=not verbose)
        
        for batch_data in pbar:
            # Non-blocking transfer to GPU
            batch_data = batch_data.to(device, non_blocking=True)
            
            # Mixed precision forward pass
            with torch.amp.autocast('cuda', enabled=use_amp and torch.cuda.is_available()):
                pred = model(batch_data)
                y = batch_data.y.float().view(-1, 1)
                loss = loss_fn(pred, y)
            
            # Backward pass with gradient scaling
            optimizer.zero_grad(set_to_none=True)  # Slightly faster than zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            # Accumulate stats
            batch_loss = loss.item()
            epoch_loss += batch_loss
            epoch_graphs += batch_data.num_graphs
            total_batches += 1
            
            # Update progress bar
            pbar.set_postfix({
                'loss': f'{batch_loss:.4f}',
                'graphs/s': f'{epoch_graphs / (time.perf_counter() - start_time):.0f}'
            })
        
        total_graphs += epoch_graphs
        epoch_losses.append(epoch_loss / len(loader))
    
    elapsed = time.perf_counter() - start_time
    
    # Collect results
    results = {
        'elapsed_time': elapsed,
        'total_graphs': total_graphs,
        'total_batches': total_batches,
        'graphs_per_sec': total_graphs / elapsed,
        'batches_per_sec': total_batches / elapsed,
        'final_loss': epoch_losses[-1] if epoch_losses else 0,
    }
    
    if torch.cuda.is_available():
        results['peak_memory_gb'] = torch.cuda.max_memory_allocated() / 1024**3
    
    return results

print("Optimized training function defined.")

In [ ]:
# =============================================================================
# BASELINE TRAINING FUNCTION (for comparison)
# =============================================================================

def train_baseline(
    model,
    graphs,
    epochs=1,
    batch_size=256,
    lr=1e-3,
    verbose=True
):
    """
    Baseline training loop matching the current models.py implementation.
    
    Uses:
    - num_workers=0 (single-threaded data loading)
    - No AMP
    - Original GraphSAGEModel with manual pooling loop
    """
    loader = DataLoader(
        graphs,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.BCELoss()
    
    total_graphs = 0
    total_batches = 0
    
    model.train()
    
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    
    start_time = time.perf_counter()
    
    for epoch in range(epochs):
        epoch_graphs = 0
        
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}", disable=not verbose)
        
        for batch_data in pbar:
            batch_data = batch_data.to(device)
            
            pred = model(batch_data)
            y = batch_data.y.float().view(-1, 1)
            loss = loss_fn(pred, y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_graphs += batch_data.num_graphs
            total_batches += 1
        
        total_graphs += epoch_graphs
    
    elapsed = time.perf_counter() - start_time
    
    results = {
        'elapsed_time': elapsed,
        'total_graphs': total_graphs,
        'graphs_per_sec': total_graphs / elapsed,
    }
    
    if torch.cuda.is_available():
        results['peak_memory_gb'] = torch.cuda.max_memory_allocated() / 1024**3
    
    return results

print("Baseline training function defined.")

In [ ]:
# =============================================================================
# HEAD-TO-HEAD COMPARISON: BASELINE VS OPTIMIZED
# =============================================================================

print("="*60)
print("TRAINING COMPARISON: Baseline vs Optimized")
print("="*60)

# Use subset for faster comparison
comparison_graphs = benchmark_graphs[:50000]
print(f"\nUsing {len(comparison_graphs):,} graphs for comparison")
print(f"Running 1 epoch each\n")

# Clear GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# --- BASELINE: Original model + original training loop ---
print("-" * 40)
print("BASELINE (original model, num_workers=0, batch_size=256)")
print("-" * 40)

baseline_model = GraphSAGEModel(
    in_channels=5, hidden_dim=128, num_layers=4, dropout=0.0, aggr='mean'
).to(device)

baseline_results = train_baseline(
    baseline_model,
    comparison_graphs,
    epochs=1,
    batch_size=256,
    lr=1e-3,
    verbose=True
)

print(f"\nBaseline: {baseline_results['graphs_per_sec']:,.0f} graphs/sec")

# Clear memory between tests
del baseline_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# --- OPTIMIZED: Optimized model + optimized training loop ---
print("\n" + "-" * 40)
print("OPTIMIZED (vectorized pooling, num_workers=4, batch_size=512, AMP)")
print("-" * 40)

optimized_model_v2 = OptimizedGraphSAGEModel(
    in_channels=5, hidden_dim=128, num_layers=4, dropout=0.0, aggr='mean'
).to(device)

optimized_results = train_optimized(
    optimized_model_v2,
    comparison_graphs,
    epochs=1,
    batch_size=512,
    lr=1e-3,
    num_workers=4,
    use_amp=True,
    use_compile=False,  # Set to True if PyTorch 2.0+ and no compilation errors
    verbose=True
)

print(f"\nOptimized: {optimized_results['graphs_per_sec']:,.0f} graphs/sec")

# Calculate speedup
speedup = optimized_results['graphs_per_sec'] / baseline_results['graphs_per_sec']

print(f"\n{'='*60}")
print(f"FINAL COMPARISON")
print(f"{'='*60}")
print(f"Baseline:  {baseline_results['graphs_per_sec']:>10,.0f} graphs/sec")
print(f"Optimized: {optimized_results['graphs_per_sec']:>10,.0f} graphs/sec")
print(f"Speedup:   {speedup:>10.2f}x")

# Estimate time savings
original_time_hours = 5.0  # Your reported ~5 hours
estimated_new_time = original_time_hours / speedup
print(f"\n--- Time Estimate ---")
print(f"Original training time: ~{original_time_hours:.1f} hours")
print(f"Estimated new time:     ~{estimated_new_time:.1f} hours")
print(f"Time saved:             ~{original_time_hours - estimated_new_time:.1f} hours")

## Section 6: Summary and Recommendations

Based on the profiling results, here are the concrete changes needed to improve GPU utilization.

In [ ]:
# =============================================================================
# SUMMARY OF FINDINGS
# =============================================================================

print("="*70)
print("SUMMARY OF GPU OPTIMIZATION FINDINGS")
print("="*70)

findings = """
┌─────────────────────────────────────────────────────────────────────┐
│                    ROOT CAUSES OF GPU UNDERUTILIZATION              │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  1. MANUAL POOLING LOOP (CRITICAL)                                  │
│     Location: models.py, GraphSAGEModel.forward()                   │
│     Problem:  Python for-loop iterating over batch indices          │
│     Solution: Replace with global_mean_pool(x, batch)               │
│     Impact:   5-20x speedup on pooling operation                    │
│                                                                     │
│  2. SINGLE-THREADED DATA LOADING                                    │
│     Location: models.py, GraphSAGE.train()                          │
│     Problem:  num_workers=0 blocks GPU while loading                │
│     Solution: Use num_workers=4 with persistent_workers=True        │
│     Impact:   2-5x speedup on data loading                          │
│                                                                     │
│  3. SMALL BATCH SIZE                                                │
│     Current:  batch_size=256                                        │
│     Problem:  GPU not fully utilized with small batches             │
│     Solution: Increase to batch_size=512 or 1024                    │
│     Impact:   1.5-3x better GPU utilization                         │
│                                                                     │
│  4. NO MIXED PRECISION                                              │
│     Problem:  Using full FP32 precision                             │
│     Solution: Enable torch.amp.autocast() + GradScaler             │
│     Impact:   1.5-2x throughput improvement                         │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
"""
print(findings)

In [ ]:
# =============================================================================
# RECOMMENDED CODE CHANGES
# =============================================================================

code_changes = '''
============================================================================
CHANGES TO MAKE IN models.py
============================================================================

1. ADD IMPORT at the top of models.py:
   ────────────────────────────────────────────────────────────────────────
   from torch_geometric.nn import global_mean_pool
   ────────────────────────────────────────────────────────────────────────

2. REPLACE manual pooling loop in GraphSAGEModel.forward() (lines 1950-1956):
   ────────────────────────────────────────────────────────────────────────
   # OLD (SLOW):
   batch_size = int(data.num_graphs) if hasattr(data, 'num_graphs') else int(batch.max().item()) + 1
   x_pooled = torch.zeros(batch_size, x.size(1), device=x.device)
   for i in range(batch_size):
       mask = (batch == i)
       if mask.sum() > 0:
           x_pooled[i] = x[mask].mean(dim=0)
   
   # NEW (FAST):
   x_pooled = global_mean_pool(x, batch)
   ────────────────────────────────────────────────────────────────────────

3. UPDATE DataLoader in GraphSAGE.train() (lines 2079-2085):
   ────────────────────────────────────────────────────────────────────────
   # OLD:
   loader = DataLoader(
       graphs,
       batch_size=batch_size,
       shuffle=True,
       num_workers=0,
       pin_memory=True,
   )
   
   # NEW:
   loader = DataLoader(
       graphs,
       batch_size=batch_size,
       shuffle=True,
       num_workers=4,
       pin_memory=True,
       persistent_workers=True,
       prefetch_factor=2,
   )
   ────────────────────────────────────────────────────────────────────────

4. ADD MIXED PRECISION in GraphSAGE.train() training loop:
   ────────────────────────────────────────────────────────────────────────
   # At the start of train():
   scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
   
   # In the training loop:
   for batch_data in loader:
       batch_data = batch_data.to(self.device, non_blocking=True)
       
       with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
           pred = self.model(batch_data)
           y = batch_data.y.float().view(-1, 1)
           loss = loss_fn(pred, y)
       
       optimizer.zero_grad(set_to_none=True)
       scaler.scale(loss).backward()
       scaler.step(optimizer)
       scaler.update()
   ────────────────────────────────────────────────────────────────────────

5. INCREASE DEFAULT BATCH SIZE in GraphSAGETrainer.__init__():
   ────────────────────────────────────────────────────────────────────────
   # OLD:
   batch_size: int = 64,
   
   # NEW:
   batch_size: int = 512,
   ────────────────────────────────────────────────────────────────────────
'''

print(code_changes)

In [ ]:
# =============================================================================
# RECOMMENDED HYPERPARAMETERS
# =============================================================================

print("="*70)
print("RECOMMENDED HYPERPARAMETERS FOR GRAPHSAGE TRAINING")
print("="*70)

recommendations = """
┌─────────────────────────────────────────────────────────────────────┐
│                    RECOMMENDED SETTINGS                              │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  DataLoader Settings:                                               │
│  ─────────────────────                                              │
│    batch_size:        512 (or 1024 if GPU memory allows)            │
│    num_workers:       4 (adjust based on CPU cores)                 │
│    pin_memory:        True                                          │
│    persistent_workers: True                                         │
│    prefetch_factor:   2                                             │
│                                                                     │
│  Training Settings:                                                 │
│  ─────────────────────                                              │
│    use_amp:           True (mixed precision)                        │
│    non_blocking:      True (async GPU transfer)                     │
│    zero_grad:         set_to_none=True (faster)                     │
│                                                                     │
│  GPU Settings (add to start of training script):                    │
│  ─────────────────────                                              │
│    torch.backends.cuda.matmul.allow_tf32 = True                     │
│    torch.backends.cudnn.allow_tf32 = True                           │
│    torch.backends.cudnn.benchmark = True                            │
│                                                                     │
│  Expected Speedup: 3-10x depending on GPU                           │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
"""
print(recommendations)

# Print final summary
if 'speedup' in dir():
    print(f"\nYour measured speedup: {speedup:.2f}x")
    print(f"Original ~5 hour training -> Estimated ~{5/speedup:.1f} hours")

In [ ]:
# =============================================================================
# READY-TO-USE: OPTIMIZED GRAPHSAGE TRAINER
# =============================================================================

class OptimizedGraphSAGETrainer:
    """
    Drop-in replacement for GraphSAGETrainer with all GPU optimizations.
    
    Copy this class to models.py or use directly from this notebook.
    
    Key optimizations:
    1. Vectorized global_mean_pool (not manual loop)
    2. Multi-worker data loading
    3. Mixed precision training (AMP)
    4. Larger default batch size
    5. Performance-tuned GPU settings
    """
    
    def __init__(
        self,
        nickname: str,
        cache_name: str = None,
        epochs: int = 10,
        batch_size: int = 512,  # Increased from 64
        lr: float = 1e-3,
        hidden_dim: int = 128,
        num_layers: int = 4,
        num_workers: int = 4,  # Multi-worker loading
        use_amp: bool = True,  # Mixed precision
        device: torch.device = None,
        base_path: Path = None,
        seed: int = None
    ):
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.base_path = base_path or BASE_PATH
        
        # Enable GPU optimizations
        if torch.cuda.is_available():
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
            torch.backends.cudnn.benchmark = True
        
        # Load cached dataset
        self._graphs = None
        if cache_name:
            cache = DatasetCache(base_path=self.base_path, device=self.device)
            cache.load(cache_name)
            self._graphs = cache.get_graphs()
        
        # Store config
        self.nickname = nickname
        self.cache_name = cache_name
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_workers = num_workers
        self.use_amp = use_amp
        self.seed = seed
        
        print(f"OptimizedGraphSAGETrainer initialized:")
        print(f"  Device: {self.device}")
        print(f"  Batch size: {batch_size}")
        print(f"  num_workers: {num_workers}")
        print(f"  AMP: {use_amp}")
    
    def train(self, verbose: bool = True):
        """Train with all optimizations enabled."""
        import random
        
        if self.seed is not None:
            torch.manual_seed(self.seed)
            np.random.seed(self.seed)
            random.seed(self.seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(self.seed)
        
        # Use optimized model with vectorized pooling
        model = OptimizedGraphSAGEModel(
            in_channels=5,
            hidden_dim=self.hidden_dim,
            num_layers=self.num_layers,
            dropout=0.0,
            aggr='mean'
        ).to(self.device)
        
        # Shuffle graphs
        graphs = self._graphs.copy()
        random.shuffle(graphs)
        
        # Optimized DataLoader
        loader = DataLoader(
            graphs,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers if self.num_workers > 0 else 0,
            pin_memory=True,
            persistent_workers=(self.num_workers > 0),
            prefetch_factor=2 if self.num_workers > 0 else None,
        )
        
        # Training setup
        optimizer = torch.optim.Adam(model.parameters(), lr=self.lr)
        loss_fn = torch.nn.BCELoss()
        scaler = torch.amp.GradScaler('cuda', enabled=self.use_amp and torch.cuda.is_available())
        
        model.train()
        
        for epoch in range(self.epochs):
            pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{self.epochs}", disable=not verbose)
            
            for batch_data in pbar:
                batch_data = batch_data.to(self.device, non_blocking=True)
                
                with torch.amp.autocast('cuda', enabled=self.use_amp and torch.cuda.is_available()):
                    pred = model(batch_data)
                    y = batch_data.y.float().view(-1, 1)
                    loss = loss_fn(pred, y)
                
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        return model

print("OptimizedGraphSAGETrainer class defined and ready to use!")

In [ ]:
# =============================================================================
# USAGE EXAMPLE
# =============================================================================

print("="*70)
print("USAGE EXAMPLE")
print("="*70)

usage = '''
# Replace your current training code with:

trainer = OptimizedGraphSAGETrainer(
    nickname="d5_baseline_seed_1",
    cache_name="d5_baseline",  # Load cached dataset
    epochs=10,
    batch_size=512,            # Larger batch size
    lr=1e-3,
    hidden_dim=128,
    num_layers=4,
    num_workers=4,             # Multi-worker loading
    use_amp=True,              # Mixed precision
    base_path=BASE_PATH,
    seed=1
)

model = trainer.train()

# Or for even faster training, try batch_size=1024 if GPU memory allows
'''
print(usage)

print("\n" + "="*70)
print("END OF DEBUG NOTEBOOK")
print("="*70)
print("\nRun this notebook on your GPU machine to identify exact bottlenecks")
print("and measure the speedup from each optimization.")